In [ ]:
sys.path.insert(0, snakemake.input.data2dd)

In [ ]:
import numpy as np
import pandas as pd
from data2dd_funcs import wrapdd

In [ ]:
year = snakemake.wildcards.year
date_range = snakemake.params.date_range
date_range = [year + "-" + date for date in date_range]
date_range = pd.date_range(date_range[0], date_range[1] + ' 23', freq='h')

In [ ]:
aggregated_regions = snakemake.params.aggregated_regions
aggregated_regions

In [ ]:
outdd = []

In [ ]:
# ev_scenarios.csv # contain the number of EVs and battery capacity per zone
ev_scenarios = snakemake.input["ev_scenarios"]

evs = pd.read_csv(ev_scenarios, index_col=(0,1)).loc[snakemake.params.ev_scenario[0], 'evs'][aggregated_regions].T.reset_index()
outdd.append(wrapdd(evs, "par_vehicles", "parameter"))

ecap = pd.read_csv(ev_scenarios, index_col=(0,1)).loc[snakemake.params.ev_scenario[0], 'ecap'][aggregated_regions].T.reset_index()
outdd.append(wrapdd(ecap, "par_ev_ecap", "parameter"))

In [ ]:
# EV_grid_connected_power_kW.csv

connected_vehicles = pd.read_csv(snakemake.input[0], index_col='date', parse_dates=True)[aggregated_regions]
connected_vehicles['day_name'] = connected_vehicles.index.day_name()
connected_vehicles['hour'] = connected_vehicles.index.hour

connected = {}

for timestamp in date_range:
    hour = timestamp.hour
    day = timestamp.day_name()
    connected[str(timestamp)] = connected_vehicles.query("day_name == @day and hour == @hour").drop(columns = ['day_name', 'hour']).reset_index(drop=True)

connected_vehicles = pd.concat(connected).reset_index(drop=True).melt(var_name='country', ignore_index=False).reset_index(names='time').set_index(['country','time']).div(1000).div(snakemake.params.ev_pcap).reset_index().astype({'time':str})
connected_vehicles['index'] = connected_vehicles['country'] + '.' + connected_vehicles['time']
connected_vehicles = connected_vehicles.drop(columns = ['country','time'])[['index','value']]
outdd.append(wrapdd(connected_vehicles, "par_grid_connected", "parameter"))

In [ ]:
# demand_driving_MWh.tsv

demand_driving = pd.read_csv(snakemake.input[1], index_col='date', parse_dates=True)[aggregated_regions]
demand_driving['day_name'] = demand_driving.index.day_name()
demand_driving['hour'] = demand_driving.index.hour

demand = {}

for timestamp in date_range:
    hour = timestamp.hour
    day = timestamp.day_name()
    demand[str(timestamp)] = demand_driving.query("day_name == @day and hour == @hour").drop(columns = ['day_name', 'hour']).reset_index(drop=True)

demand_driving = pd.concat(demand).reset_index(drop=True).melt(var_name='country', ignore_index=False).reset_index(names='time').set_index(['country','time']).div(1000).reset_index().astype({'time':str})
demand_driving['index'] = demand_driving['country'] + '.' + demand_driving['time'] + '.EV'
demand_driving = demand_driving.drop(columns = ['country','time'])[['index','value']]
outdd.append(wrapdd(demand_driving, "par_driving_demand", "parameter"))

In [ ]:
# demand_ev_charging_MWh.tsv
demand_ev_charging = pd.read_csv(snakemake.input[2], index_col='date', parse_dates=True)[aggregated_regions]
demand_ev_charging['day_name'] = demand_ev_charging.index.day_name()
demand_ev_charging['hour'] = demand_ev_charging.index.hour

charging = {}

for timestamp in date_range:
    hour = timestamp.hour
    day = timestamp.day_name()
    charging[str(timestamp)] = demand_ev_charging.query("day_name == @day and hour == @hour").drop(columns = ['day_name', 'hour']).reset_index(drop=True)

demand_ev_charging = pd.concat(charging).reset_index(drop=True).melt(var_name='country', ignore_index=False).reset_index(names='time').set_index(['country','time']).div(1000).reset_index().astype({'time':str})
demand_ev_charging['index'] = demand_ev_charging['country'] + '.' + demand_ev_charging['time']
demand_ev_charging = demand_ev_charging.drop(columns = ['country','time'])[['index','value']]
outdd.append(wrapdd(demand_ev_charging, "par_ev_charging", "parameter"))

In [ ]:
outdd = np.concatenate(outdd, axis=0)

In [ ]:
np.savetxt(snakemake.output[0], outdd, delimiter=" ", fmt="%s")